# 4 Agents on a Governed Endpoint

**Purpose:**
Build a small tool-using agent that runs on the governed endpoint from Sections 1 to 3, so that every step it takes is metered, rate-limited, and budgeted by the LLM Gateway.

**What we're doing**

We write an agent as ordinary client code: a loop that asks the model what to do, runs a local tool, hands the result back, and repeats until the model answers. Every pass through the loop is one chat completion through the gateway. We give the loop its own guardrails (a step limit, a token tally, and a clean stop on a gateway rejection), then rerun it under Section 3's rate limit and wallet budget and watch the gateway govern it.

**Why we're doing it**

* An agent turns one question into many model calls. That multiplies cost, and it is exactly the traffic that rate limits and wallet budgets exist for
* An agent that does not understand "no" from the gateway retries forever, or crashes halfway through a task. A governed agent stops cleanly and says why
* Everything you set up in Section 3 applies to the agent without any change, because the agent is just another gateway client

**What to look for**

* Whether your vLLM endpoint returns native tool calls, or needs the plain-prompt JSON fallback
* The trace table: one row per step, the tool it ran, and the tokens it used
* The agent waiting out a rate limit and finishing, and stopping at a wallet budget instead of looping
* The agent's requests in the **AI Usage** dashboard

**Output**

A function `run_agent()` that completes a short multi-step task through the gateway, prints a step-by-step trace, and stops cleanly at its step limit, its token limit, or a gateway rejection.

> **Before you start:** You need Section 3's `PROFILE`, `GATEWAY_MODEL`, and, if you recorded them, the two rejection fingerprints from 3.5 and 3.6. Section 4.3 may need a Klique administrator to change your vLLM Model Deployment. Section 4.8 reuses Section 3's rate limit and wallet budget, with the same warning: use a throwaway user or project for the budget.

## 4.1 What the Docs Cover, and What This Notebook Adds

The Klique docs describe several agent features. None of them is a general agent runtime that you program from a notebook, so the agent in this notebook is client code. Here is where each part of the story comes from.

| Piece | Documented? | Where |
|---|---|---|
| Tool calling on a vLLM Model Deployment: **Enable Auto Tool Choice** and **Tool Call Parser** | Yes | [vLLM Model Deployment](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment) |
| Which tool-call parser fits a Qwen3 model | **Not documented by Klique.** The page links to the vLLM tool-calling docs. The custom-image page's example uses `--tool-call-parser hermes`. | [vLLM Model Deployment (Custom Image)](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment_custom_image) |
| Whether the LLM Gateway passes `tools` and `tool_calls` through unchanged | **Not documented: observe and record** (4.4) | |
| Rate-limit and wallet rejections are both `429` | Yes, in the API reference | [routers API](https://klique.ai/docs/latest/docs/references/api/routers) |
| Every gateway request is metered and shown in AI Usage | Yes | [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage) |
| Routing requests that declare tools to a chosen Resource (`content.has_tools`) | Yes, in the API reference | [routers API](https://klique.ai/docs/latest/docs/references/api/routers) |
| Gateway guardrails: deny and allow patterns | Yes | [LLM Gateway settings](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway#guardrails) |
| An agent loop, its step limit, its token limit, and its trace | **The author's own code.** Klique does not document an agent loop. | |
| Agentic coding sessions, the Klique MCP server, and MCP Server Deployment | Yes. They are Klique's own agent features (see 4.10). | [Agentic Interface](https://klique.ai/docs/latest/docs/agentic_interface/) |

Two documented facts shape the code:

* **Without Enable Auto Tool Choice, tool calls fail.** The docs say it lets vLLM accept `tool_choice: "auto"` requests, and that without it, tool-calling clients "fail with a 400" error about auto tool choice. It also requires a **Tool Call Parser** that matches the model's output format.
* **A client may not be able to tell a rate limit from a budget by status code.** The API reference says a resource's rate limit is "rejected with 429", and that an exhausted wallet "answers the wallet's 429". That is why 4.2 relies on the fingerprints you recorded in Section 3, and treats an unrecognized `429` as ambiguous.

## 4.2 Load the Token, the Gateway URL, and the Section 3 Helpers

This cell is the setup from Section 3. Notebooks do not import from each other, so the helpers are **repeated from Section 3** in compact form:

* `send_chat()` is Section 3's `send()`, extended to take a full message list and an optional list of tools. Like `send()`, it does not raise on an HTTP error.
* `describe()`, `fingerprint()`, `classify()`, `wait_time()`, and `GovernanceError` are unchanged in what they do.

Paste the two fingerprints you printed in 3.5 and 3.6 into `RATE_LIMIT_FP` and `BUDGET_FP`. If you did not record them, leave them as `None`: every `429` is then treated as **ambiguous**, retried a few times, and then stops the agent. The token is shown by its last four characters only.

In [ ]:
import sys
import ast
import json
import time
import random
import operator
from datetime import datetime, timezone
sys.path.insert(0, "..")
from config import KLIQUE_GATEWAY_TOKEN as token, KLIQUE_LLM_GATEWAY_URL as gateway_url
import requests

PROFILE = "frank-test"                 # Your LLM Gateway profile
GATEWAY_MODEL = "vllm-qwen-franktest"  # Your Section 1 endpoint's name in that profile

RATE_LIMIT_FP = None   # From 3.5, for example (429, "rate_limit_exceeded")
BUDGET_FP = None       # From 3.6

print(f"LLM Gateway:   {gateway_url}")
print(f"Profile:       {PROFILE}")
print(f"Gateway model: {GATEWAY_MODEL}")
print(f"Token:         ...{token[-4:]}")

UPSTREAM_STATUSES = {502, 503, 504}


class GovernanceError(Exception):
    """The gateway refused the request in a way that retrying will not fix."""


def send_chat(messages, model_name=None, tools=None, tool_choice=None, max_tokens=512, timeout=120):
    """Send one chat completion (a full message list) through the gateway. Returns the raw Response."""
    body = {"model": model_name or GATEWAY_MODEL, "messages": messages, "max_tokens": max_tokens}
    if tools:
        body["tools"] = tools
        if tool_choice:
            body["tool_choice"] = tool_choice
    return requests.post(
        f"{gateway_url}/v1/chat/completions",
        headers={"Authorization": f"Bearer {token}", "X-ClearML-Profile": PROFILE},
        json=body,
        timeout=timeout,
    )


def describe(resp):
    """Print what a client sees: status, limit-related headers, and the start of the body."""
    print(f"HTTP {resp.status_code} {resp.reason}")
    for name, value in resp.headers.items():
        if any(k in name.lower() for k in ("limit", "retry", "remaining", "reset", "quota", "budget")):
            print(f"  header {name}: {value}")
    print(f"  body: {resp.text[:400]}")


def fingerprint(resp):
    """Reduce a rejection to (status code, error code or type)."""
    label = None
    try:
        body = resp.json()
    except ValueError:
        body = None
    if isinstance(body, dict):
        err = body.get("error")
        if isinstance(err, dict):
            label = err.get("code") or err.get("type")
        elif isinstance(err, str):
            label = err
    return (resp.status_code, label)


def classify(resp):
    """Label a response: ok, rate_limit, budget, ambiguous, upstream, or other."""
    if resp.ok:
        return "ok"
    fp = fingerprint(resp)
    rl, bd = RATE_LIMIT_FP, BUDGET_FP
    if rl and bd and rl == bd and fp == rl:
        return "ambiguous"
    if rl and fp == tuple(rl):
        return "rate_limit"
    if bd and fp == tuple(bd):
        return "budget"
    if resp.status_code == 429:
        return "ambiguous"   # Both limits answer 429 (API reference), so an unknown 429 could be either
    if resp.status_code in UPSTREAM_STATUSES:
        return "upstream"
    return "other"


def wait_time(resp, attempt, base_delay):
    """Seconds to wait: Retry-After if it is a number, else exponential backoff with jitter."""
    header = resp.headers.get("Retry-After") if resp is not None else None
    if header is not None:
        try:
            return float(header)
        except ValueError:
            pass
    return base_delay * (2 ** attempt) + random.uniform(0, 1)

## 4.3 Turn On Tool Calling for Your vLLM Endpoint

*An administrator may need to do this step.*

Native tool calling is a setting on the vLLM Model Deployment, not on the gateway. In your Section 1 instance's configuration, under **Deployed Models**:

1. Turn on **Enable Auto Tool Choice**.
2. Set **Tool Call Parser** to the parser that matches your model's tool-call output. The Klique docs do not say which parser fits which model. They point to the vLLM tool-calling docs.
3. Relaunch the instance so the new settings take effect.

If your instance uses **Config override**, the wizard settings above are ignored: the docs say the override "replaces the wizard configuration entirely". Add the vLLM flags to the override instead. The custom-image app's docs show them as `--enable-auto-tool-choice --tool-call-parser hermes`.

**You can skip this step.** If you cannot change the deployment, the notebook still works: 4.4 detects that native tool calls are not available, and the agent uses a plain-prompt JSON fallback instead.

> **Reasoning models:** Qwen3 thinks in `<think>` tags by default. If **Enable Reasoning** and a **Reasoning Parser** are set, the docs say the reasoning is parsed "into OpenAI API format", so it should arrive separately from the answer. Otherwise it arrives inside the reply text. The agent in 4.6 strips `<think>` blocks either way.

## 4.4 Define the Tools and Probe the Endpoint

The agent gets three small tools. They are local, read-only, and safe, so a confused model cannot do damage:

| Tool | What it does |
|---|---|
| `calculator` | Evaluates arithmetic (`+ - * / ** %` and parentheses) without `eval()` |
| `clock` | Returns the current UTC time |
| `lookup` | Reads one entry from a small, made-up GPU inventory |

`TOOLS` describes them in the OpenAI-compatible `tools` format, for native tool calling. `TOOL_FUNCTIONS` maps each name to the Python function that runs it.

In [ ]:
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.Mod: operator.mod, ast.USub: operator.neg, ast.UAdd: operator.pos}


def calculator(expression):
    """Evaluate plain arithmetic safely: numbers and + - * / ** % only."""
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            if isinstance(node.op, ast.Pow) and abs(ev(node.right)) > 10:
                raise ValueError("exponent too large")
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("only arithmetic is allowed")
    return ev(ast.parse(str(expression), mode="eval"))


def clock():
    """Current time in UTC."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")


INVENTORY = {   # Made-up teaching data, not from Klique
    "lab-a100": {"gpus": 8, "gpu_type": "A100 80GB", "hourly_rate_usd_per_gpu": 2.5},
    "lab-l40s": {"gpus": 4, "gpu_type": "L40S", "hourly_rate_usd_per_gpu": 1.2},
}


def lookup(key):
    """Read one entry from INVENTORY."""
    return INVENTORY.get(str(key), {"error": f"unknown key {key!r}", "known_keys": sorted(INVENTORY)})


TOOL_FUNCTIONS = {"calculator": calculator, "clock": clock, "lookup": lookup}

TOOLS = [
    {"type": "function", "function": {
        "name": "calculator", "description": "Evaluate an arithmetic expression, for example '8 * 2.5 * 3'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "clock", "description": "Get the current date and time in UTC.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "lookup", "description": "Look up a GPU pool in the inventory. Keys: " + ", ".join(sorted(INVENTORY)) + ".",
        "parameters": {"type": "object", "properties": {"key": {"type": "string"}}, "required": ["key"]}}},
]


def run_tool(name, arguments):
    """Run one tool by name. Errors are returned to the model as text, never raised."""
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return {"error": f"unknown tool {name!r}"}
    try:
        return {"result": fn(**(arguments or {}))}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

Now probe the endpoint once: send a question with `tools` and `tool_choice: "auto"`, and look at what comes back. The Klique docs do not say whether the gateway passes `tools` through unchanged, so this cell records what yours does:

* **HTTP 200 with `tool_calls`** in the message: native tool calling works through the gateway. `TOOL_MODE` becomes `"native"`.
* **HTTP 400** mentioning auto tool choice: the deployment does not have **Enable Auto Tool Choice** on (see 4.3). `TOOL_MODE` becomes `"json"`.
* **HTTP 200 without `tool_calls`**: the request was accepted, but the model's tool call was not parsed. The parser may not match the model. `TOOL_MODE` becomes `"json"`.

In [ ]:
def detect_tool_mode(resp):
    """Return ("native" or "json", reason) from one probe response."""
    if not resp.ok:
        return "json", f"HTTP {resp.status_code}: native tools refused"
    message = resp.json()["choices"][0]["message"]
    if message.get("tool_calls"):
        return "native", "the reply carried tool_calls"
    return "json", "HTTP 200 but no tool_calls in the reply"

In [ ]:
probe = send_chat(
    [{"role": "user", "content": "What is 17 * 23? Use the calculator tool."}],
    tools=TOOLS, tool_choice="auto", max_tokens=512,
)
describe(probe)
TOOL_MODE, reason = detect_tool_mode(probe)
print(f"\nTOOL_MODE = {TOOL_MODE!r} ({reason})")

Write down what you saw: the status code, and if it was `200`, the `tool_calls` entry. You can force a mode by setting `TOOL_MODE` yourself, for example to compare the two.

> **Checkpoint:** `TOOL_MODE` is set. If it is `"json"` and you expected native tool calls, check 4.3 and the troubleshooting table.

## 4.5 A Governed Step

One agent step is one gateway request. `governed_step()` is Section 3's `chat_with_limits()` reshaped for an agent: it takes the whole conversation, and it reports how many HTTP attempts the step took. It covers one model, because a fallback alias would change the agent's model in the middle of a task.

* **Success:** returns the parsed reply and the number of attempts.
* **Rate limit** or **ambiguous `429`:** waits and retries, up to `max_retries`.
* **Budget:** raises `GovernanceError` at once. Waiting does not clear a wallet budget.
* **Unreachable model:** retries once, then raises `GovernanceError`.
* **Anything else** (`400`, `401`, `404`, a guardrail rejection): raises `GovernanceError` with the status and the start of the body. The docs do not say what status a guardrail rejection returns.

In [ ]:
def governed_step(messages, tools=None, max_tokens=512, max_retries=3, base_delay=2.0):
    """One agent step through the gateway. Returns (reply_json, attempts) or raises GovernanceError."""
    attempt = 0

    def stop(message):
        err = GovernanceError(message)
        err.attempts = attempt + 1   # HTTP attempts made in this step, for the trace
        return err

    while True:
        resp = None
        try:
            resp = send_chat(messages, tools=tools, tool_choice="auto" if tools else None, max_tokens=max_tokens)
            kind = classify(resp)
        except requests.RequestException:
            kind = "upstream"

        if kind == "ok":
            return resp.json(), attempt + 1
        if kind == "budget":
            raise stop("Wallet budget reached. Retrying will not help until it resets or is raised.")
        if kind == "other":
            raise stop(f"Rejected: HTTP {resp.status_code} {resp.text[:200]}")

        limit = max_retries if kind in ("rate_limit", "ambiguous") else 1
        if attempt >= limit:
            raise stop(f"Still rejected after {attempt} retries ({kind}).")
        delay = wait_time(resp, attempt, base_delay)
        print(f"    {kind}: waiting {delay:.1f}s before retry {attempt + 1}")
        time.sleep(delay)
        attempt += 1

## 4.6 The Agent Loop

`run_agent()` repeats one cycle: ask the model, run the tool it chose, add the result to the conversation, ask again. It works in either mode from 4.4:

* **`native`:** the model's reply carries `tool_calls`. The loop runs each one and answers with a `tool` message, as the OpenAI-compatible format expects.
* **`json`:** no `tools` are sent. A system prompt asks the model to reply with one JSON object, either `{"tool": ..., "arguments": {...}}` or `{"final": ...}`. The loop strips any `<think>` block, finds the first JSON object in the reply, and feeds the tool result back as a user message.

The guardrails are the author's own choices, not Klique settings:

| Guardrail | Default | Stops the run with status |
|---|---|---|
| Step limit | `max_steps=6` | `max_steps` |
| Token tally from each reply's `usage` | `max_run_tokens=6000` | `token_limit` |
| Unparseable replies in a row (JSON mode) | `max_parse_errors=2` | `parse_error` |
| Gateway rejection that retrying will not fix | | `stopped_by_gateway` |

The token limit is a client-side safety net and counts tokens. Klique's wallet budget counts dollars and is enforced by the gateway. They are separate.

In [ ]:
JSON_SYSTEM_PROMPT = (
    "You are a careful assistant that can use tools. Available tools:\n"
    + "\n".join(f"- {t['function']['name']}: {t['function']['description']} "
                 f"Arguments: {json.dumps(t['function']['parameters'].get('properties', {}))}" for t in TOOLS)
    + "\nReply with exactly one JSON object and nothing else. To use a tool: "
      '{"tool": "<name>", "arguments": {...}}. To give the final answer: {"final": "<answer>"}.'
      " Use one tool per reply."
)


def strip_think(text):
    """Remove <think>...</think> blocks (and an unclosed <think> tail) from a reply."""
    text = text or ""
    while "<think>" in text:
        start = text.index("<think>")
        end = text.find("</think>", start)
        text = text[:start] + ("" if end == -1 else text[end + len("</think>"):])
    return text.replace("</think>", "").strip()


def first_json_object(text):
    """Return the first JSON object found in text, or None."""
    decoder = json.JSONDecoder()
    for i, ch in enumerate(text):
        if ch == "{":
            try:
                obj, _ = decoder.raw_decode(text[i:])
            except ValueError:
                continue
            if isinstance(obj, dict):
                return obj
    return None


def run_agent(task, mode=None, max_steps=6, max_run_tokens=6000, max_parse_errors=2, max_tokens=512, verbose=True):
    """Run a tool-using agent through the governed gateway. Never raises GovernanceError: it stops and reports."""
    mode = mode or TOOL_MODE
    if mode == "native":
        messages = [{"role": "user", "content": task}]
    else:
        messages = [{"role": "system", "content": JSON_SYSTEM_PROMPT}, {"role": "user", "content": task}]
    trace, tokens, parse_errors = [], {"prompt": 0, "completion": 0}, 0
    status, answer = "max_steps", None

    for step in range(1, max_steps + 1):
        row = {"step": step, "mode": mode, "tool": "", "requests": 0, "prompt_tokens": 0,
               "completion_tokens": 0, "note": ""}
        trace.append(row)
        try:
            reply, row["requests"] = governed_step(messages, tools=TOOLS if mode == "native" else None,
                                                   max_tokens=max_tokens)
        except GovernanceError as e:
            row["requests"] = getattr(e, "attempts", 0)
            row["note"] = f"stopped: {e}"
            status = "stopped_by_gateway"
            break

        usage = reply.get("usage") or {}
        row["prompt_tokens"] = usage.get("prompt_tokens", 0)
        row["completion_tokens"] = usage.get("completion_tokens", 0)
        tokens["prompt"] += row["prompt_tokens"]
        tokens["completion"] += row["completion_tokens"]
        message = reply["choices"][0]["message"]

        if mode == "native" and message.get("tool_calls"):
            messages.append({"role": "assistant", "content": message.get("content") or "",
                             "tool_calls": message["tool_calls"]})
            names = []
            for call in message["tool_calls"]:
                fn = call.get("function", {})
                try:
                    args = json.loads(fn.get("arguments") or "{}")
                except ValueError:
                    args = None
                result = run_tool(fn.get("name"), args) if isinstance(args, dict) else {"error": "arguments were not valid JSON"}
                names.append(fn.get("name", "?"))
                messages.append({"role": "tool", "tool_call_id": call.get("id", ""), "content": json.dumps(result)})
            row["tool"] = ", ".join(names)
            row["note"] = json.dumps(result)[:60]
        elif mode == "native":
            answer = strip_think(message.get("content"))
            row["note"] = "final answer"
            status = "done"
        else:
            text = strip_think(message.get("content"))
            obj = first_json_object(text)
            if obj is not None and "final" in obj:
                answer = str(obj["final"])
                row["note"] = "final answer"
                status = "done"
            elif obj is not None and "tool" in obj:
                parse_errors = 0
                result = run_tool(obj["tool"], obj.get("arguments") if isinstance(obj.get("arguments"), dict) else {})
                row["tool"] = str(obj["tool"])
                row["note"] = json.dumps(result)[:60]
                messages.append({"role": "assistant", "content": text})
                messages.append({"role": "user", "content": f"TOOL RESULT ({obj['tool']}): {json.dumps(result)}"})
            else:
                parse_errors += 1
                row["note"] = "could not parse a JSON object"
                if parse_errors >= max_parse_errors:
                    status = "parse_error"
                else:
                    messages.append({"role": "assistant", "content": text})
                    messages.append({"role": "user", "content": "Reply with exactly one JSON object, as instructed."})

        if verbose:
            print(f"step {step}: {row['tool'] or '-'} {row['note']}")
        if status in ("done", "parse_error"):
            break
        if tokens["prompt"] + tokens["completion"] > max_run_tokens:
            status = "token_limit"
            break

    return {"status": status, "answer": answer, "trace": trace, "tokens": tokens}


def print_trace(run):
    """Print one row per step, then the totals."""
    cols = ["step", "mode", "tool", "requests", "prompt_tokens", "completion_tokens", "note"]
    print(" | ".join(cols))
    for r in run["trace"]:
        print(" | ".join(str(r[c]) for c in cols))
    t = run["tokens"]
    print(f"\nstatus: {run['status']}   steps: {len(run['trace'])}   "
          f"requests: {sum(r['requests'] for r in run['trace'])}   "
          f"tokens: {t['prompt']} in, {t['completion']} out")
    if run["answer"]:
        print(f"answer: {run['answer']}")

## 4.7 Run the Agent

The task needs two tools: a lookup, then a calculation. A good run takes three steps: `lookup`, `calculator`, then the final answer.

In [ ]:
TASK = ("Using the inventory, how much does it cost to run every GPU in the 'lab-a100' pool for 3 hours? "
        "Look up the pool first, then calculate.")

run = run_agent(TASK)
print()
print_trace(run)

Read the trace. Each row is one gateway request, unless **requests** is above 1, which means the step was retried. **prompt_tokens** grows with every step, because each request carries the whole conversation so far. That growth is the cost of an agent: three steps cost much more than three times one short question.

Try the other mode too. Run `run_agent(TASK, mode="json")`, or `mode="native"` if 4.4 found native tools, and compare the token totals.

> **Checkpoint:** the run ends with `status: done` and an answer of $60 (8 GPUs × $2.50 × 3 hours). If it ends with `max_steps` or `parse_error`, read the **note** column. Smaller models often need the JSON fallback, or a few more steps.

## 4.8 Watch the Gateway Govern the Agent

*An administrator does the settings in this step, as in Section 3.*

**Step limit.** Run the agent with a single step. It cannot finish, so it stops with `max_steps` instead of looping:

In [ ]:
print_trace(run_agent(TASK, max_steps=1, verbose=False))

**Rate limit.** Put back Section 3.5's low rate limit, for example 5 requests per 60 seconds, on the same profile or Resource. Then run the agent three times in a row. The later steps should be rejected, wait in `governed_step()`, and continue. The **requests** column shows the retries. If the window is longer than the retries cover, the run stops with `stopped_by_gateway` and the message "Still rejected", rather than hanging.

In [ ]:
for i in range(3):
    print(f"--- run {i + 1} ---")
    print_trace(run_agent(TASK, verbose=False))
    print()

**Clear the rate limit before the budget test**, as in Section 3.

**Wallet budget.** Repeat Section 3.6's setup: **Enforce wallet budgets** on, a small allocated budget on a **throwaway** user or project, and a token that belongs to it. Run the agent in a loop until the budget runs out. The run that crosses it should stop with `stopped_by_gateway` and the wallet message after zero retries, if `BUDGET_FP` is set and differs from `RATE_LIMIT_FP`. If the two are the same, or not set, the agent retries a few times and then stops with "Still rejected".

> **Warning:** as in Section 3.6, the docs say a budget that has been crossed rejects further requests "until the next calendar month, unless enforcement is turned off." Use a throwaway user or project.

In [ ]:
MAX_RUNS = 30
for i in range(1, MAX_RUNS + 1):
    result = run_agent(TASK, verbose=False)
    t = result["tokens"]
    print(f"run {i:>2}: {result['status']:<18} tokens {t['prompt'] + t['completion']}")
    if result["status"] == "stopped_by_gateway":
        print()
        print_trace(result)
        break

> **Checkpoint:** the budget run stopped with `stopped_by_gateway`, and the trace shows the step where it stopped. The agent did not keep calling.

**Undo it now.** Turn **Enforce wallet budgets** off, or delete the allocated budget.

## 4.9 Find the Agent in AI Usage

Open the **AI Usage** tab of the Admin Dashboard, as in Section 3.3:

1. Set the source to **Gateway**.
2. Set the time range to cover this notebook's runs.
3. Group the **Usage Breakdown** by **model**, then by **user**.

Compare the **In** and **Out** token cards with the token totals your traces printed. Each agent step is a separate request. If your Resource has pricing from Section 3.4, the **Cost** card shows what the agent cost.

The docs do not describe a way to tag a request as coming from an agent, or to group usage by a run. To tell agent traffic apart in the dashboard, run the agent as a dedicated user, or under its own project. The dashboard groups usage by user and by project.

The docs also do not say whether a rejected request appears in AI Usage. **Not documented: observe and record** whether the retries from 4.8 show up.

> **Checkpoint:** the dashboard's token totals for the last hour include your agent's runs.

See the [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage).

## 4.10 Klique's Own Agent Features

The agent in this notebook is client code. Klique also documents agent features of its own. This section points to them, using only the documented steps.

**Agentic coding sessions.** The Claude Code, Codex, OpenCode, and Pi apps launch an agent CLI in a browser-based VS Code. The docs say a session "routes its calls through the LLM Gateway using the gateway Profile you select", which "lets administrators enforce wallet budgets and rate limits on it". They also warn: "Forcing direct access to the provider with your own API key bypasses the gateway." These are governed agents with no code to write. Note that these clients send `tool_choice: "auto"`, so a vLLM Resource behind them needs **Enable Auto Tool Choice** (4.3). See [Launching an Agentic Coding Session](https://klique.ai/docs/latest/docs/getting_started/launching_agentic_session).

**The Klique MCP server.** It lets an agent such as Claude Code or Cursor operate Klique itself (projects, tasks, models, apps) in natural language. It has its own host and a `/mcp` URL from your administrator. It uses a different Bearer token from the gateway token in this notebook. The docs give a credential-free health check: `GET /health` answers `200` with a `status` of `ok`. The optional cell below runs only that check. See [Agentic Interface](https://klique.ai/docs/latest/docs/agentic_interface/).

**MCP Server Deployment.** An app that runs MCP servers (Filesystem, PostgreSQL, Fetch, and others) as containers behind the AI Application Gateway, with request and latency plots in its dashboard. See [MCP Server Deployment](https://klique.ai/docs/latest/docs/webapp/applications/apps_mcp_server).

**Routing and guardrails for tool traffic.** The API reference lists `content.has_tools` as a routing-rule field, so an administrator can send requests that declare tools to a chosen Resource. The settings page gives "the requested model or max output tokens" as examples of what a rule can route on. **Not documented:** whether the WebApp offers `has_tools`. Content guardrails: the NeMo Guardrails app's **execution rails** "control the inputs and outputs of custom actions or tools the model calls." See the [routers API](https://klique.ai/docs/latest/docs/references/api/routers) and [Guardrails](https://klique.ai/docs/latest/docs/governance_engine/guardrails).

In [ ]:
MCP_SERVER_URL = None   # Optional: "https://<your-mcp-server-domain-name>", from your administrator

if MCP_SERVER_URL:
    r = requests.get(f"{MCP_SERVER_URL.rstrip('/')}/health", timeout=15)
    print(f"GET /health: HTTP {r.status_code}  {r.text[:200]}")
else:
    print("Set MCP_SERVER_URL to check the Klique MCP server's health.")

## 4.11 Put Everything Back

| Setting | Undo |
|---|---|
| **Enable Auto Tool Choice** and **Tool Call Parser** (4.3) | Keep them if you want tool calling. They only take effect for requests that send tools |
| Rate limit (4.8) | Clear it, as in Section 3.9 |
| **Enforce wallet budgets** and the allocated budget (4.8) | Turn enforcement off, or delete the allocated budget |

## Troubleshooting

| Symptom | Likely cause |
|---|---|
| Probe returns `400` mentioning auto tool choice | **Enable Auto Tool Choice** is off, or there is no **Tool Call Parser**. See 4.3, or use `TOOL_MODE = "json"`. |
| Probe returns `200` but no `tool_calls` | The **Tool Call Parser** does not match the model's output format, or a **Config override** replaced the wizard settings. |
| Changed the deployment settings and nothing happened | The instance must be relaunched. With a **Config override**, put the flags in the override. |
| JSON mode ends with `parse_error` | The model did not reply with a JSON object. Raise `max_tokens` so the `<think>` block does not use up the reply, or try native mode. |
| The run ends with `max_steps` | The model kept calling tools. Read the **note** column, and raise `max_steps` only if the steps are making progress. |
| `stopped_by_gateway` with "Still rejected" | The rate limit's window is longer than the retries cover, or a budget `429` that the client could not recognize. See Section 3.5 and 3.6. |
| `stopped_by_gateway` with `HTTP 404` | The alias is not in your profile. See Section 2's troubleshooting. |
| `stopped_by_gateway` with another status | Possibly a gateway guardrail pattern. The docs do not give its status code. Ask your administrator. |

## Wrap-Up

The agent is one more client of the gateway:

```
Agent loop (your code) → LLM Gateway → vLLM Model Endpoint
     └ step limit, token tally ┘   └ rate limit, wallet budget, metering ┘
```

Everything Section 3 built governs it without any change: every step is metered, a rate limit slows it down, and a wallet budget stops it.

## Related Docs

- [vLLM Model Deployment: tool calling and reasoning settings](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment)
- [vLLM Model Deployment (Custom Image)](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment_custom_image)
- [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway)
- [LLM Gateway settings](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway)
- [AI Quotas: wallet budgets](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_ai_quotas)
- [routers API reference: rate limits, wallet 429, routing rules](https://klique.ai/docs/latest/docs/references/api/routers)
- [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)
- [Launching an Agentic Coding Session](https://klique.ai/docs/latest/docs/getting_started/launching_agentic_session)
- [Agentic Interface (Klique MCP server)](https://klique.ai/docs/latest/docs/agentic_interface/)
- [MCP Server Deployment](https://klique.ai/docs/latest/docs/webapp/applications/apps_mcp_server)
- [Guardrails](https://klique.ai/docs/latest/docs/governance_engine/guardrails)